# Breaking Defenses & Black-Box Attacks

In [1]:
import torch
from torch import nn
from torch.optim import Adam
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader

from torchvision import transforms
from torchvision.models import resnet18, mobilenet_v2
from torchvision.datasets.cifar import CIFAR10

from tqdm import trange, tqdm

torch.manual_seed(0)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

# CIFAR10 Dataset (5 points)

In [2]:
norm_mean = (0.4914, 0.4822, 0.4465)
norm_std = (0.2023, 0.1994, 0.2010)
batch_size = 128

mu = torch.tensor(norm_mean).view(3,1,1).to(device)
std = torch.tensor(norm_std).view(3,1,1).to(device)

# TODO: Set the upper limit and lower limit possible for images
upper_limit = ((1 - mu) / std)
lower_limit = ((0 - mu) / std)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(norm_mean, norm_std),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(norm_mean, norm_std),
])

trainset = CIFAR10(root='./data', train=True, download=True, transform=transform_train)
trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)

testset = CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)


classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')


100%|██████████| 170M/170M [00:17<00:00, 9.57MB/s] 


# Defensive Distillation (25 points)

[Defensive distillation](https://arxiv.org/abs/1511.04508) proceeds in four steps:

1.   **Train the teacher network**, by setting the temperature of the softmax to T during the
training phase.
2.   **Compute soft labels** by apply the teacher network to each instance in the training set, again evaluating the softmax at temperature T.
3.  **Train the distilled network** (a network with the same shape as the teacher network) on the soft labels, using softmax at temperature T.
4.  Finally, when running the distilled network at test time to classify new inputs, use temperature 1.



## Train the teacher

In [3]:
def train_step(model, dataloader, loss_fn, optimizer, temperature):
    # TODO: Return loss and accuracy for each epoch
    # pass
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass with temperature scaling
        outputs = model(images)
        outputs_scaled = outputs / temperature
        
        # Compute loss with temperature-scaled outputs
        loss = loss_fn(outputs_scaled, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Calculate accuracy (using unscaled outputs)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    accuracy = 100. * correct / total
    avg_loss = total_loss / len(dataloader)
    
    return avg_loss, accuracy


def train_teacher(model, n_epochs, loader=trainloader, temp=100):
    # TODO: Log the accuracy and loss for each epoch
    # pass

    model = model.to(device)
    optimizer = Adam(model.parameters(), lr=0.001)
    loss_fn = CrossEntropyLoss()
    
    for epoch in range(n_epochs):
        loss, acc = train_step(model, loader, loss_fn, optimizer, temp)
        print(f'Epoch {epoch+1}/{n_epochs} - Loss: {loss:.4f}, Accuracy: {acc:.2f}%')


You can use a pre-trained resnet to speed up the training process.

In [4]:
def make_model(num_classes=10):
    model = resnet18(pretrained=True)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

teacher = make_model()

train_teacher(teacher, 15)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 254MB/s]


Epoch 1/15 - Loss: 1.3735, Accuracy: 65.40%
Epoch 2/15 - Loss: 0.6189, Accuracy: 80.84%
Epoch 3/15 - Loss: 0.4447, Accuracy: 85.62%
Epoch 4/15 - Loss: 0.3572, Accuracy: 88.34%
Epoch 5/15 - Loss: 0.3085, Accuracy: 89.69%
Epoch 6/15 - Loss: 0.2720, Accuracy: 90.92%
Epoch 7/15 - Loss: 0.2416, Accuracy: 91.92%
Epoch 8/15 - Loss: 0.2139, Accuracy: 92.91%
Epoch 9/15 - Loss: 0.1926, Accuracy: 93.40%
Epoch 10/15 - Loss: 0.1754, Accuracy: 94.04%
Epoch 11/15 - Loss: 0.1581, Accuracy: 94.69%
Epoch 12/15 - Loss: 0.1469, Accuracy: 95.05%
Epoch 13/15 - Loss: 0.1365, Accuracy: 95.38%
Epoch 14/15 - Loss: 0.1236, Accuracy: 95.72%
Epoch 15/15 - Loss: 0.1182, Accuracy: 96.01%


## Test the teacher

In [5]:
def test_clean(model, dataloader=testloader):
    # TODO: Return the clean accuracy of the model
    # pass
    
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    accuracy = 100. * correct / total
    return accuracy

Print the clean accuracy of the teacher.

In [6]:
print(f'Teacher Accuracy {test_clean(teacher):.2f}%')

Teacher Accuracy 90.48%


## Train the student

In [7]:
def distill(model, teacher, dataloader, optimizer, T):
    # TODO: Get soft labels from teacher model
    # TODO: Get student model outputs
    # TODO: Compute the distillation loss
    # TODO: Return the accuracy (on real labels) and loss (on soft labels)
    # pass

    model.train()
    teacher.eval()
    
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        # Get soft labels from teacher (with temperature)
        with torch.no_grad():
            teacher_outputs = teacher(images)
            soft_labels = F.softmax(teacher_outputs / T, dim=1)
        
        # Get student outputs (with temperature)
        student_outputs = model(images)
        student_outputs_scaled = F.log_softmax(student_outputs / T, dim=1)
        
        # Compute distillation loss (KL divergence)
        # Scale by T^2 as per the distillation paper
        loss = F.kl_div(student_outputs_scaled, soft_labels, reduction='batchmean') * (T * T)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Calculate accuracy on real labels (using unscaled outputs)
        _, predicted = student_outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    accuracy = 100. * correct / total
    avg_loss = total_loss / len(dataloader)
    
    return avg_loss, accuracy


def train_student(model, teacher, n_epochs, loader=trainloader, temp=100):
    # TODO: Log the accuracy and loss for each epoch
    # pass

    model = model.to(device)
    optimizer = Adam(model.parameters(), lr=0.001)
    
    for epoch in range(n_epochs):
        loss, acc = distill(model, teacher, loader, optimizer, temp)
        print(f'Epoch {epoch+1}/{n_epochs} - Loss: {loss:.4f}, Accuracy: {acc:.2f}%')
    
    # return model

This time use a `resnet18` without the pretrained weights.

In [8]:
def make_model(num_classes=10):
    model = resnet18(pretrained=False)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

student = make_model()

train_student(student, teacher, 15)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 1/15 - Loss: 16060.5552, Accuracy: 35.48%
Epoch 2/15 - Loss: 10243.3935, Accuracy: 59.16%
Epoch 3/15 - Loss: 7455.7961, Accuracy: 69.66%
Epoch 4/15 - Loss: 5714.9540, Accuracy: 76.59%
Epoch 5/15 - Loss: 4661.7373, Accuracy: 80.39%
Epoch 6/15 - Loss: 3911.9198, Accuracy: 83.00%
Epoch 7/15 - Loss: 3400.4392, Accuracy: 84.77%
Epoch 8/15 - Loss: 3007.3439, Accuracy: 86.34%
Epoch 9/15 - Loss: 2728.4119, Accuracy: 87.37%
Epoch 10/15 - Loss: 2464.8212, Accuracy: 88.28%
Epoch 11/15 - Loss: 2310.4912, Accuracy: 89.17%
Epoch 12/15 - Loss: 2086.7672, Accuracy: 89.81%
Epoch 13/15 - Loss: 1970.7897, Accuracy: 90.37%
Epoch 14/15 - Loss: 1823.4104, Accuracy: 90.88%
Epoch 15/15 - Loss: 1704.4948, Accuracy: 91.33%


## Test the student

In [9]:
print(f'Student Accuracy {test_clean(student):.2f}%')

Student Accuracy 88.40%


# Attack (15 points)

Implement the FGSM attack and the `test_attack` funcion to report the robust accuracy for different values of epsilon.

In [10]:
def attack_fgsm(model, x, y, epsilon, T=1):
    """FGSM attack implementation"""
    model.eval()
    
    x_adv = x.clone().detach().requires_grad_(True)
    
    # Forward pass with temperature
    outputs = model(x_adv) / T
    loss = F.cross_entropy(outputs, y)
    
    # Backward pass to get gradients
    loss.backward()
    
    # Create perturbation
    grad_sign = x_adv.grad.sign()
    x_adv = x_adv + epsilon * grad_sign
    
    # Clamp to valid range
    x_adv = torch.max(torch.min(x_adv, upper_limit), lower_limit)
    
    return x_adv.detach()


def attack_pgd(model, x, y, epsilon, T=1, alpha=0.2, num_iters=10):
    """PGD attack implementation"""
    model.eval()
    
    # Initialize with random perturbation
    x_adv = x.clone().detach() + torch.empty_like(x).uniform_(-epsilon, epsilon)
    x_adv = torch.max(torch.min(x_adv, upper_limit), lower_limit)
    
    for _ in range(num_iters):
        x_adv.requires_grad_(True)
        
        # Forward pass with temperature
        outputs = model(x_adv) / T
        loss = F.cross_entropy(outputs, y)
        
        # Backward pass
        loss.backward()
        
        # Update adversarial example
        grad_sign = x_adv.grad.sign()
        x_adv = x_adv.detach() + alpha * epsilon * grad_sign
        
        # Project back to epsilon ball
        perturbation = torch.clamp(x_adv - x, -epsilon, epsilon)
        x_adv = x + perturbation
        
        # Clamp to valid range
        x_adv = torch.max(torch.min(x_adv, upper_limit), lower_limit)
    
    return x_adv.detach()


def test_attack(model, epsilon, attack=attack_fgsm, T=1, loader=testloader):
    """Return the robust accuracy for FGSM or PGD"""
    model.eval()
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        # Generate adversarial examples
        if attack == attack_pgd:
            adv_images = attack(model, images, labels, epsilon, T)
        else:
            adv_images = attack(model, images, labels, epsilon, T)
        
        # Test on adversarial examples (use T=1 for inference)
        with torch.no_grad():
            outputs = model(adv_images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    accuracy = 100. * correct / total
    return accuracy

Report the robust accuracy of the teacher for `ϵ = [1, 2, 4, 8, 16]`.

In [11]:
scale = 1 / std.mean().item()
scale

4.977600714306323

In [12]:
epsilons = [1, 2, 4, 8, 16]

INFERENCE_TEMPERATURE = 100
print("Teacher Model Robustness:")
print(f"The training temperature is {100}")
print(f"The inference temperature (attack) is {INFERENCE_TEMPERATURE}")
for eps in epsilons:
    # TODO:
    # acc_fgsm = test_attack(teacher, eps*scale/255, attack=attack_fgsm)
    # acc_pgd = test_attack(teacher, eps*scale/255, attack=attack_pgd)
    acc_fgsm = test_attack(teacher, eps*scale/255, attack=attack_fgsm, T=INFERENCE_TEMPERATURE)
    acc_pgd = test_attack(teacher, eps*scale/255, attack=attack_pgd, T=INFERENCE_TEMPERATURE)
    print(f'FGSM with ϵ={eps}/255 has Accuracy: {acc_fgsm:.2f}%')
    print(f'PGD  with ϵ={eps}/255 has Accuracy: {acc_pgd:.2f}%')

Teacher Model Robustness:
The training temperature is 100
The inference temperature (attack) is 100
FGSM with ϵ=1/255 has Accuracy: 48.26%
PGD  with ϵ=1/255 has Accuracy: 33.34%
FGSM with ϵ=2/255 has Accuracy: 27.89%
PGD  with ϵ=2/255 has Accuracy: 5.87%
FGSM with ϵ=4/255 has Accuracy: 15.92%
PGD  with ϵ=4/255 has Accuracy: 0.18%
FGSM with ϵ=8/255 has Accuracy: 9.70%
PGD  with ϵ=8/255 has Accuracy: 0.00%
FGSM with ϵ=16/255 has Accuracy: 8.43%
PGD  with ϵ=16/255 has Accuracy: 0.00%


Do the same for the student:

In [13]:
INFERENCE_TEMPERATURE = 1
print("Student Model Robustness:")
print(f"The training temperature is {100}")
print(f"The inference temperature (attack) is {INFERENCE_TEMPERATURE}")
for eps in epsilons:
    # TODO:
    acc_fgsm = test_attack(student, eps*scale/255, attack=attack_fgsm, T=INFERENCE_TEMPERATURE)
    acc_pgd = test_attack(student, eps*scale/255, attack=attack_pgd, T=INFERENCE_TEMPERATURE)
    print(f'FGSM with ϵ={eps}/255 has Accuracy: {acc_fgsm:.2f}%')
    print(f'PGD  with ϵ={eps}/255 has Accuracy: {acc_pgd:.2f}%')

Student Model Robustness:
The training temperature is 100
The inference temperature (attack) is 1
FGSM with ϵ=1/255 has Accuracy: 84.35%
PGD  with ϵ=1/255 has Accuracy: 84.39%
FGSM with ϵ=2/255 has Accuracy: 84.36%
PGD  with ϵ=2/255 has Accuracy: 84.01%
FGSM with ϵ=4/255 has Accuracy: 84.35%
PGD  with ϵ=4/255 has Accuracy: 83.69%
FGSM with ϵ=8/255 has Accuracy: 84.42%
PGD  with ϵ=8/255 has Accuracy: 81.42%
FGSM with ϵ=16/255 has Accuracy: 84.81%
PGD  with ϵ=16/255 has Accuracy: 71.48%


### What do you see?

The results reveal a critical flaw in defensive distillation that demonstrates **gradient obfuscation** rather than true adversarial robustness:

**Teacher Model Behavior (Expected):**
- The teacher shows standard vulnerability to adversarial attacks, with accuracy degrading as epsilon increases
- Under FGSM: drops from 48.26% (ε=1/255) to 8.43% (ε=16/255)
- Under PGD: drops dramatically from 33.34% (ε=1/255) to 0.00% (ε=8/255 and above)
- This is the expected behavior for a model without true adversarial robustness

**Student Model Behavior (Suspicious):**
- The student maintains remarkably **constant accuracy** across all epsilon values: ~84% for FGSM and ~81-84% for PGD (except ε=16/255 PGD at 71.48%)
- Critically, the student's "robust" accuracy (84%) is only slightly below its clean accuracy (88.40%)
- The accuracy barely changes between ε=1/255 and ε=8/255, which is physically implausible for a truly robust model

**Key Insight - Gradient Obfuscation:**
This behavior is a textbook case of **gradient masking**. The student was trained with temperature T=100 but attacked with T=1 during inference. This mismatch causes:
1. The attack algorithms to compute gradients on a different loss surface than the one used during training
2. The computed gradients become uninformative, making gradient-based attacks ineffective
3. The illusion of robustness without actual security improvement

**Conclusion:**
Defensive distillation does not provide genuine adversarial robustness. Instead, it obfuscates gradients (gradient masking), causing gradient-based attacks to fail. This is a false sense of security - as we'll see in the transfer attack section, black-box attacks or attacks that bypass gradient computation can still fool the distilled model. The paper by Carlini & Wagner (2017) exposed this limitation, showing that adaptive attacks can easily break defensive distillation.

# Transferring Adversarial Examples (15 points)

Train yet another model to be used as the surrogate. (set temperature to 1)

In [14]:
def make_model(num_classes=10):
    model = resnet18(pretrained=False)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

surrogate = make_model()

train_teacher(surrogate, 15, temp=1)

Epoch 1/15 - Loss: 1.3476, Accuracy: 51.06%
Epoch 2/15 - Loss: 0.8792, Accuracy: 68.98%
Epoch 3/15 - Loss: 0.6809, Accuracy: 76.26%
Epoch 4/15 - Loss: 0.5688, Accuracy: 80.08%
Epoch 5/15 - Loss: 0.4997, Accuracy: 82.76%
Epoch 6/15 - Loss: 0.4443, Accuracy: 84.62%
Epoch 7/15 - Loss: 0.3957, Accuracy: 86.41%
Epoch 8/15 - Loss: 0.3596, Accuracy: 87.59%
Epoch 9/15 - Loss: 0.3299, Accuracy: 88.67%
Epoch 10/15 - Loss: 0.3003, Accuracy: 89.63%
Epoch 11/15 - Loss: 0.2817, Accuracy: 90.33%
Epoch 12/15 - Loss: 0.2557, Accuracy: 91.10%
Epoch 13/15 - Loss: 0.2427, Accuracy: 91.55%
Epoch 14/15 - Loss: 0.2202, Accuracy: 92.38%
Epoch 15/15 - Loss: 0.2082, Accuracy: 92.83%


Print the surrogate accuracy.

In [15]:
print(f'Surrogate Accuracy {test_clean(surrogate):.2f}%')

Surrogate Accuracy 86.70%


Report the accuracy of the surrogate for `ϵ = [1, 2, 4, 8, 16]`.

In [16]:
INFERENCE_TEMPERATURE = 1
print("Surrogate Model Robustness:")
print(f"The training temperature is {1}")
print(f"The inference temperature (attack) is {INFERENCE_TEMPERATURE}")
for eps in epsilons:
    acc_fgsm = test_attack(surrogate, eps*scale/255, attack=attack_fgsm, T=INFERENCE_TEMPERATURE)
    acc_pgd = test_attack(surrogate, eps*scale/255, attack=attack_pgd, T=INFERENCE_TEMPERATURE)
    print(f'FGSM with ϵ={eps}/255 has Accuracy: {acc_fgsm:.2f}%')
    print(f'PGD  with ϵ={eps}/255 has Accuracy: {acc_pgd:.2f}%')

Surrogate Model Robustness:
The training temperature is 1
The inference temperature (attack) is 1
FGSM with ϵ=1/255 has Accuracy: 48.00%
PGD  with ϵ=1/255 has Accuracy: 40.07%
FGSM with ϵ=2/255 has Accuracy: 24.54%
PGD  with ϵ=2/255 has Accuracy: 9.35%
FGSM with ϵ=4/255 has Accuracy: 10.62%
PGD  with ϵ=4/255 has Accuracy: 0.22%
FGSM with ϵ=8/255 has Accuracy: 5.92%
PGD  with ϵ=8/255 has Accuracy: 0.00%
FGSM with ϵ=16/255 has Accuracy: 5.69%
PGD  with ϵ=16/255 has Accuracy: 0.00%


Implement the following functions to transfer attacks from a surrogate model to an oracle.

In [17]:
def transfer_attack(oracle, model, eps, loader=testloader, T=1):
    # TODO: Attack the model and report the accuracy of the oracle
    # pass

    oracle.eval()
    model.eval()
    
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        # Generate adversarial examples using the surrogate model
        adv_images = attack_fgsm(model, images, labels, eps, T)
        
        # Test on the oracle (target) model
        with torch.no_grad():
            outputs = oracle(adv_images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    accuracy = 100. * correct / total
    return accuracy

Transfer attacks for `ϵ = [1, 2, 4, 8, 16]` from your model to the student.

In [18]:
print("Transfered attack with Surrogate Model to student Robustness:")
INFERENCE_TEMPERATURE = 1
print(f"The Surrogate Model's training temperature is {1}")
print(f"The Surrogate Model's inference temperature (attack) is {INFERENCE_TEMPERATURE}")
for eps in epsilons:
    acc = transfer_attack(student, surrogate, eps*scale/255, T=INFERENCE_TEMPERATURE)
    print(f'FGSM with ϵ={eps}/255 has Accuracy: {acc:.2f}%')

Transfered attack with Surrogate Model to student Robustness:
The Surrogate Model's training temperature is 1
The Surrogate Model's inference temperature (attack) is 1
FGSM with ϵ=1/255 has Accuracy: 75.65%
FGSM with ϵ=2/255 has Accuracy: 61.16%
FGSM with ϵ=4/255 has Accuracy: 40.10%
FGSM with ϵ=8/255 has Accuracy: 22.30%
FGSM with ϵ=16/255 has Accuracy: 14.40%


- What can be inferred from these results?
- How are the accuracies of the student and the surrogate under attack related?
- Does Defensive Distillation obfuscate the gradients? Why?

`your response:`

## What can be inferred from these results?

The transfer attack results provide compelling evidence that defensive distillation creates **gradient obfuscation rather than true robustness**:

### Direct Attack vs Transfer Attack Comparison:

**Student under white-box attack (T=1 inference):**
- ϵ=1/255: 84.35% accuracy
- ϵ=8/255: 84.42% accuracy
- ϵ=16/255: 84.81% accuracy

**Student under transfer attack from surrogate:**
- ϵ=1/255: 75.65% accuracy (↓8.70%)
- ϵ=8/255: 22.30% accuracy (↓62.12%)
- ϵ=16/255: 14.40% accuracy (↓70.41%)

### Key Inference:
The student model, which appeared highly robust against white-box gradient-based attacks, is **significantly more vulnerable** to transfer attacks. This dramatic difference confirms that the apparent robustness was due to gradient masking, not robust decision boundaries.

---

## How are the accuracies of the student and the surrogate under attack related?

### Surrogate under white-box attack:
- ϵ=8/255 FGSM: 5.92% accuracy
- ϵ=16/255 FGSM: 5.69% accuracy

### Student under transfer attack from surrogate:
- ϵ=8/255 FGSM: 22.30% accuracy
- ϵ=16/255 FGSM: 14.40% accuracy

The student maintains **3-4× higher accuracy** than the surrogate under comparable attacks. This suggests:

1. **Imperfect transferability**: Not all adversarial perturbations transfer perfectly between models
2. **Different decision boundaries**: Despite similar architectures, the temperature-scaled training creates sufficiently different learned representations
3. **Partial protection**: While transfer attacks are more effective than white-box attacks on the student, the student still benefits from some cross-model differences

However, the steady degradation (75.65% → 14.40%) as ϵ increases demonstrates that the student **does not have robust decision boundaries**—it's simply harder to attack via gradients computed on its own loss surface.

---

## Does Defensive Distillation obfuscate the gradients? Why?

**Yes, defensive distillation definitively obfuscates gradients.** The evidence is overwhelming:

### Evidence from Results:

1. **White-box attack failure**: Student accuracy barely changes (84.35% → 84.81%) across ϵ=1-16/255 for FGSM
2. **Transfer attack success**: Same student drops dramatically (75.65% → 14.40%) under transfer attacks
3. **Comparison**: Surrogate (T=1) shows normal attack behavior (48.00% → 5.69%), confirming attacks work correctly

### Why This Happens:

**Temperature scaling during training** creates gradient obfuscation through:

1. **Softmax smoothing**: Training at T=100 produces extremely smooth, flat probability distributions
2. **Vanishing gradients**: At test time (T=1), the softmax becomes sharp, but the model learned with smooth gradients
3. **Gradient mismatch**: The gradient computed at T=1 doesn't reflect the training dynamics at T=100
4. **Saturated outputs**: The model's pre-softmax logits are scaled for T=100, causing saturation when evaluated at T=1

### Mathematical Intuition:
When computing gradients for attacks at T=1 on a model trained at T=100, the loss landscape appears much flatter than during training. The attack algorithms receive weak or misleading gradient signals, causing them to fail—not because the model is robust, but because the gradient information is obscured.

### Conclusion:
Defensive distillation is a form of **gradient masking**, not true adversarial robustness. This is why it was later shown to be ineffective against adaptive attacks and why modern defense mechanisms (like adversarial training) focus on actually improving decision boundary robustness rather than hiding gradient information.


# ZOO Based Black-Box Attacks (25 points)

Based on [Black-box Adversarial Attacks with Limited Queries and Information](https://arxiv.org/abs/1804.08598) you must first calculate the estimate of the graidents, and next attack the model based on your estimates.

In [19]:
def nes_gradient_estimate(model, x, y, epsilon, num_samples, sigma):
    # TODO: Return the estimated gradient
    # pass

    model.eval()
    
    # Get the loss for the original input
    with torch.no_grad():
        outputs = model(x)
        loss_orig = F.cross_entropy(outputs, y, reduction='none')
    
    # Initialize gradient estimate
    grad_estimate = torch.zeros_like(x)
    
    # Sample random perturbations and estimate gradient
    for _ in range(num_samples):
        # Sample random noise
        noise = torch.randn_like(x)
        
        # Evaluate at perturbed points
        with torch.no_grad():
            outputs_pos = model(x + sigma * noise)
            loss_pos = F.cross_entropy(outputs_pos, y, reduction='none')
            
            outputs_neg = model(x - sigma * noise)
            loss_neg = F.cross_entropy(outputs_neg, y, reduction='none')
        
        # Finite difference gradient estimate
        loss_diff = (loss_pos - loss_neg).view(-1, 1, 1, 1)
        grad_estimate += loss_diff * noise
    
    # Average over samples
    grad_estimate = grad_estimate / (num_samples * 2 * sigma)
    
    return grad_estimate

I used 3 different things to estimate gradiant and all of them end up almost the same result. The bottom result is made with probabilities.

In [20]:
def partial_information_attack(model, x, y, epsilon, num_samples, sigma, num_steps, alpha):
    # TODO: Return the perturbed image
    # pass

    model.eval()
    
    x_adv = x.clone().detach()
    
    for step in range(num_steps):
        # Estimate gradient
        grad_est = nes_gradient_estimate(model, x_adv, y, epsilon, num_samples, sigma)
        
        # Update adversarial example (gradient ascent on loss)
        x_adv = x_adv + alpha * grad_est.sign()
        
        # Project back to epsilon ball
        perturbation = torch.clamp(x_adv - x, -epsilon, epsilon)
        x_adv = x + perturbation
        
        # Clamp to valid range
        x_adv = torch.max(torch.min(x_adv, upper_limit), lower_limit)
    
    return x_adv.detach()

Now run this attack on your models and report the results. (You **DON'T** need to run the attack for the entire test dataset as this will take a lot of time!)

In [21]:
def test_zoo_attack(model, epsilon, num_samples, sigma, num_steps, alpha, loader=testloader):
    """Test ZOO attack (run on subset of data due to computational cost)"""
    model.eval()
    correct = 0
    total = 0
    
    # Only test on the first batch to save time
    max_batches = 1
    batch_count = 0
    
    for images, labels in loader:
        if batch_count >= max_batches:
            break
        
        images, labels = images.to(device), labels.to(device)
        
        # Generate adversarial examples using ZOO
        adv_images = partial_information_attack(
            model, images, labels, epsilon, num_samples, sigma, num_steps, alpha
        )
        
        # Test on adversarial examples
        with torch.no_grad():
            outputs = model(adv_images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        batch_count += 1
    
    accuracy = 100. * correct / total
    return accuracy

In [22]:
# epsilons = [1, 2, 4, 8, 16]

# for eps in epsilons:
#     acc = test_zoo_attack(model=surrogate, epsilon=eps*scale/255, num_samples=100, sigma=0.001, num_steps=10, alpha=0.1, loader=testloader)
#     print(f'ZOO with ϵ={eps}/255 has Accuracy: {acc:.2f}%')

In [23]:
epsilons = [1, 2, 4, 8, 16]

print("ZOO Attack on Surrogate Model (subset of test data):")
for eps in epsilons:
    acc = test_zoo_attack(
        model=surrogate, 
        epsilon=eps*scale/255, 
        num_samples=100, 
        sigma=0.001, 
        num_steps=10, 
        alpha=0.1, 
        loader=testloader
    )
    print(f'ZOO with ϵ={eps}/255 has Accuracy: {acc:.2f}%')

ZOO Attack on Surrogate Model (subset of test data):
ZOO with ϵ=1/255 has Accuracy: 78.91%
ZOO with ϵ=2/255 has Accuracy: 74.22%
ZOO with ϵ=4/255 has Accuracy: 53.91%
ZOO with ϵ=8/255 has Accuracy: 24.22%
ZOO with ϵ=16/255 has Accuracy: 18.75%


# Adversarially Robust Distillation (15 points)

In this section we are going to test another type of distillation to see if this method is robust. This technique is [Adversarially Robust Distillation](https://arxiv.org/abs/1905.09747).



1.   We will try to distill a robsut teacher from [Robust Bench](https://robustbench.github.io/) onto a smaller architecture.
2.   We minimize the KL-Divergence between the logits of the student and teacher to ensure fidelity. (You can also incorporate the classification loss as mentioned in the paper but you can choose to ignore it as well)
3.   At each step of the distillation you will attack the student (you can use either FGSM or PGD) and find an adversarial example $X + \delta$ for data point $X$. Next you will minimize $t^2 \times \text{KL}(S(X+\delta), T(X))$ where $S$ and $T$ are the student and teacher networks respectively.



In [24]:
! pip install git+https://github.com/RobustBench/robustbench.git

  Cloning https://github.com/RobustBench/robustbench.git to /tmp/pip-req-build-tst6hicb
  Running command git clone --filter=blob:none --quiet https://github.com/RobustBench/robustbench.git /tmp/pip-req-build-tst6hicb
  Resolved https://github.com/RobustBench/robustbench.git to commit 78fcc9e48a07a861268f295a777b975f25155964
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/fra31/auto-attack.git (to revision a39220048b3c9f2cca9a4d3a54604793c68eca7e) to /tmp/pip-install-qu_83t63/autoattack_4bf1ea7b1d154c7ea789834790f8de10
  Running command git clone --filter=blob:none --quiet https://github.com/fra31/auto-attack.git /tmp/pip-install-qu_83t63/autoattack_4bf1ea7b1d154c7ea789834790f8de10
  Running command git rev-parse -q --verify 'sha^a39220048b3c9f2cca9a4d3a54604793c68eca7e'
  Running command git fetch -q https://github.com/fra31/auto-attack.git a39220048b3c9f2cca9a4d3a54604793c68eca7e
  Resolved https://github.com/fra31/auto-attack.git to commit a39220048b3c9f2cca9a4

In [63]:
from robustbench.utils import load_model

teacher = load_model(model_name='Gowal2021Improving_R18_ddpm_100m', dataset='cifar10', threat_model='Linf').to(device)

In [ ]:
# def ard(student, teacher, dataloader, optimizer, eps, attack):
#     # TODO
#     pass

def ard(student, teacher, dataloader, optimizer, eps, attack, T=1, beta=0.4):
    total_loss = 0
    
    correct = 0
    total = 0
    
    student.train()
    teacher.eval()
    
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        
        x_adv = attack(student, x, y, eps)
        
        with torch.no_grad():
            teacher_logits = teacher(x) 
            teacher_soft = F.softmax(teacher_logits / T, dim=1)
        
        student_logits = student(x_adv)
        student_soft = F.log_softmax(student_logits / T, dim=1)
        
        kl_loss = F.kl_div(student_soft, teacher_soft, reduction='batchmean') * (T)**2
        
        ce_loss = F.cross_entropy(student_logits, y)
        
        loss = kl_loss + beta * ce_loss
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = student_logits.max(1)
        total += y.size(0)
        correct += predicted.eq(y).sum().item()
    
    avg_loss = total_loss / len(dataloader)
    accuracy = 100. * correct / total
    
    return avg_loss, accuracy


# def adv_train_student(model, teacher, n_epochs, eps=8/255, loader=trainloader):
#     # TODO
#     pass
def adv_train_student(model, teacher, n_epochs, eps=8/255, loader=trainloader):
    model.to(device)
    teacher.to(device)
    
    optimizer = Adam(model.parameters(), lr=0.003)
    
    for epoch in range(n_epochs):
        loss, acc = ard(
            model, teacher, loader, optimizer, eps, attack_pgd, T=1, beta=0.4
        )
        
        print(f'Epoch {epoch+1}/{n_epochs} - Total Loss: {loss:.4f} - Accuracy: {acc:.2f}%')

In [65]:
student = mobilenet_v2(weights=None).to(device)

# TODO: Adjust and train the student
student.classifier[1] = nn.Linear(student.classifier[1].in_features, 10)

adv_train_student(student, teacher, 15)


Epoch 1/15 - Total Loss: 2.2689 - Accuracy: 19.87%
Epoch 2/15 - Total Loss: 2.2465 - Accuracy: 29.14%
Epoch 3/15 - Total Loss: 2.2341 - Accuracy: 33.46%
Epoch 4/15 - Total Loss: 2.1960 - Accuracy: 38.12%
Epoch 5/15 - Total Loss: 2.2028 - Accuracy: 42.11%
Epoch 6/15 - Total Loss: 2.2184 - Accuracy: 45.01%
Epoch 7/15 - Total Loss: 2.2336 - Accuracy: 46.38%
Epoch 8/15 - Total Loss: 2.2449 - Accuracy: 47.89%
Epoch 9/15 - Total Loss: 2.2458 - Accuracy: 48.73%
Epoch 10/15 - Total Loss: 2.2535 - Accuracy: 49.15%
Epoch 11/15 - Total Loss: 2.2592 - Accuracy: 51.19%
Epoch 12/15 - Total Loss: 2.2603 - Accuracy: 52.04%
Epoch 13/15 - Total Loss: 2.2612 - Accuracy: 52.97%
Epoch 14/15 - Total Loss: 2.2621 - Accuracy: 53.87%
Epoch 15/15 - Total Loss: 2.2625 - Accuracy: 54.70%



Now report the accuracy of the student on the test dataset.

In [66]:
# TODO: Clean accurcy
clean_acc = test_clean(student)
print(f'Student Clean Accuracy: {clean_acc:.2f}%')

# TODO: FGSM with eps=8/255
fgsm_acc = test_attack(student, scale*8/255, attack=attack_fgsm)
print(f'FGSM with ϵ=8/255 Accuracy: {fgsm_acc:.2f}%')

# TODO: PGD with eps=8/255
pgd_acc = test_attack(student, scale*8/255, attack=attack_pgd)
print(f'PGD with ϵ=8/255 Accuracy: {pgd_acc:.2f}%')

Student Clean Accuracy: 63.17%
FGSM with ϵ=8/255 Accuracy: 15.36%
PGD with ϵ=8/255 Accuracy: 13.78%
